<a href="https://colab.research.google.com/github/msiplab/VieWork/blob/master/code/python/vie_sec09_transform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://matlab.mathworks.com/open/github/v1?repo=msiplab/VieWork&file=code/matlab/vie_sec09_transform.m&project=VieWork.prj" target="_parent"><img src="https://www.mathworks.com/images/responsive/global/open-in-matlab-online.svg" alt="Open in MATLAB Online"/></a>

# 第9回 線形変換処理
村松正吾　「画像情報工学」

動作確認：Python 3.12（Google Colab），NumPy，SciPy，scikit-image，Matplotlib

MATLAB のライブスクリプト `code/matlab/vie_sec09_transform.m` を Python に移植したもの。第9回のスライド（vie2026-09）で使う図と数値を計算する。自然画像の隣接画素の相関，信号変換 $\mathbf{y}=\mathbf{A}\mathbf{x}$ と基底ベクトル $\mathbf{b}_m$（$\mathbf{B}=\mathbf{A}^{-1}$ の列），基底画像，離散コサイン変換（DCT），離散ハール変換（DHT），ブロック処理，ハール変換のフィルタバンク表現を扱う。

スライドの図と数値は MATLAB 版で作っている。新潟大学の学生は[大学の包括ライセンス](https://jp.mathworks.com/academia/tah-portal/niigata-university-31576302.html)で MATLAB を使えるので，MATLAB 版もぜひ動かしてほしい。上の「Open in MATLAB Online」のボタンを押すと，ブラウザだけで MATLAB Online が開き，VieWork が自分の MATLAB Drive に複製されて，このノートブックに対応するライブスクリプトが開く（インストールは不要）。さらに [MATLAB Connector](https://jp.mathworks.com/products/matlab-drive.html) を手元の PC に入れると，MATLAB Drive のファイルが PC のフォルダと自動で同期されるので，MATLAB Online で作業したファイルを手元の MATLAB でもそのまま開ける。

記号は教科書（村松正吾『多次元信号・画像処理の基礎と展開』）に合わせる。教科書の例・例題（例「分散共分散行列」，例題「二次元DCT」「二次元DHT」「ブロック処理」）は，ここで計算し直して教科書の解答と一致することを確かめる。図の配色は MSIP Lab のロゴの 3 色（緑 #008855，青 #2E75B6，橙 #C55A11）と灰色を使う。

画像処理には scikit-image，SciPy，NumPy を使う。教科書の例題の数値はスライドと一致することを `assert` で確かめている。画像から計算する値は，ライブラリの実装が MATLAB と異なるため，スライドの値とわずかに異なることがある。

$2\times2$ の数値例では，ハール変換（$\pi/4$ 回転）行列

$$\mathbf{A}=\frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\-1&1\end{pmatrix}$$

を用いる（教科書の $\mathbf{H}_2=\mathbf{C}_2$ とは第 2 行の符号だけが異なる）。

## 準備
共通ヘルパー `vie.py`（サンプル画像の取得，日本語フォントと配色の設定）を読み込む。Colab では GitHub から取得する。

In [ ]:
import importlib
import os
import sys
import urllib.request

# Colab では毎回 GitHub から最新の vie.py を取得する（VieWork を clone した場合は同じフォルダの vie.py を使う）
if "google.colab" in sys.modules or not os.path.exists("vie.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/msiplab/VieWork/master/code/python/vie.py", "vie.py")

import numpy as np
import matplotlib.pyplot as plt
from scipy import fft, signal
from skimage.exposure import rescale_intensity
from skimage.metrics import peak_signal_noise_ratio
from skimage.transform import resize
from skimage.util import img_as_float, view_as_blocks
import vie

vie = importlib.reload(vie)   # 取得し直した vie.py を読み込み直す

# 図の配色はスライドと揃え，ロゴの 3 色（メインの緑，寒色系の青，暖色系の橙）と灰色を使う
cmain, ccool, cwarm, cgray = vie.CMAIN, vie.CCOOL, vie.CWARM, vie.CGRAY
np.set_printoptions(precision=4, suppress=True, linewidth=120)


def show(ax, X, title=None):
    """画像を軸なし・縦横比 1 で表示する（グレースケールは 0 を黒，最大値（255 か 1）を白）．"""
    vmax = 255 if X.dtype == np.uint8 else 1
    ax.imshow(X, cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()
    if title is not None:
        ax.set_title(title)

In [ ]:
A = np.array([[1, 1], [-1, 1]]) / np.sqrt(2)
B = np.linalg.inv(A)                           # = A^T（直交行列）
print("A =\n", A)
print("B =\n", B)

## 自然画像の隣接画素の散布図
画像は教科書のサンプル画像 msipimg04.tif（石造りの建物）を $256\times256$ 画素のグレースケールに縮小して使う（共通関数 `vie.msipimg`）。暗い空などの平坦な領域と石の細かな模様を含み，相関の強さと変換によるスパース化がはっきり見える（石像の顔 msipimg05 やモンブラン msipimg07 と比べて選んだ）。

この画像の水平方向の隣接画素ペア $\mathbf{x}^{[i]}=(x_0\ x_1)^\top$（重ならないように 2 画素ずつ）を散布図にする。添え字は第09回全体で 0 始まりに揃える（教科書の図 6.1 は 1 始まり）。ほぼ対角線上に並ぶ：隣どうしは似た値をとる（相関が強い）。

In [ ]:
X = img_as_float(vie.msipimg(4, 256, "gray"))   # 石造りの建物（256×256）
x1 = X[:, 0::2]
x2 = X[:, 1::2]
P = np.vstack([x1.ravel(order="F"), x2.ravel(order="F")])   # 2×S の画素ペア
S = P.shape[1]
print("S =", S)

標本分散共分散行列は教科書の式どおり $1/S$ で正規化する（`np.cov` の `bias=True`）。

$$\hat{\boldsymbol{\Sigma}}_\mathcal{X}=\frac{1}{S}\sum_{i=1}^{S}\underline{\mathbf{x}}^{[i]}(\underline{\mathbf{x}}^{[i]})^\top,\quad \underline{\mathbf{x}}^{[i]}=\mathbf{x}^{[i]}-\hat{\boldsymbol{\mu}}_\mathcal{X}$$

In [ ]:
Sx = np.cov(P, bias=True)
print("Sx =\n", Sx)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 4.4), layout="constrained")
show(axs[0], X, "msipimg04（256×256）")
axs[1].scatter(P[0], P[1], s=3, color=ccool, alpha=0.3, edgecolors="none")
axs[1].set(xlim=(0, 1), ylim=(0, 1), aspect="equal", xticks=np.arange(0, 1.01, 0.2),
           yticks=np.arange(0, 1.01, 0.2), xlabel="$x_0$", ylabel="$x_1$")
axs[1].grid(True)
plt.show()

## 信号変換の効果
教科書の例「分散共分散行列」と同じ変換行列（$\mathbf{C}_2=\mathbf{H}_2$，2 点のユニタリ DFT 行列 $\mathbf{W}_2/\sqrt{2}$ と一致）

$$\mathbf{A}_\mathrm{ex}=\frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$$

で $\mathbf{y}^{[i]}=\mathbf{A}_\mathrm{ex}\mathbf{x}^{[i]}$ と変換すると，$y_0$（和）の方向に大きく広がり，$y_1$（差）の方向の広がりは小さい。分散共分散行列は $\mathbf{A}_\mathrm{ex}\hat{\boldsymbol{\Sigma}}_\mathcal{X}\mathbf{A}_\mathrm{ex}^\top$ となり，非対角成分（相関）がほぼ 0 になる。

In [ ]:
Aex = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
Q = Aex @ P
Sy = np.cov(Q, bias=True)
Sy_check = Aex @ Sx @ Aex.T                    # 同じ値になる
print("Sy =\n", Sy)
print("Sy_check =\n", Sy_check)

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.4), layout="constrained")
ax.scatter(Q[0], Q[1], s=3, color=ccool, alpha=0.3, edgecolors="none")
ax.set(xlim=(0, 1.5), ylim=(-0.75, 0.75), aspect="equal", xticks=np.arange(0, 1.51, 0.5),
       yticks=[-0.5, 0, 0.5], xlabel="$y_0$", ylabel="$y_1$")
ax.grid(True)
plt.show()

$y_1$（差分）の多くは 0 付近に集まる：変換係数の絶対値が 0.02 未満の割合

In [ ]:
ratio = np.mean(np.abs(Q[1]) < 0.02)
print(f"ratio = {ratio:.4f}（{100 * ratio:.0f} %）")

### 教科書の例「分散共分散行列」を再現する
教科書の図 6.1 は，図 1.2 (a) の画像（MsipWorkM の msipimg01.tif を $96\times96$ 画素に縮小したもの）の水平隣接画素ペアから作られている。同じ手順で計算し，教科書の $\hat{\boldsymbol{\Sigma}}_\mathcal{X}$，$\hat{\boldsymbol{\Sigma}}_\mathcal{Y}$ と比べる。画像は共通関数 `vie.msipimg` で読み込み，scikit-image の `resize`（双線形補間，アンチエイリアスあり）で縮小する。

この例は画像の縮小を含むので，結果は縮小の実装に依存する。縮小の実装が教科書（MATLAB の `imresize`）と異なるため，小数第 4 位で教科書の値と 0.0001 ほど異なる要素がある（そのため `assert` ではなく，表示して比べる）。

In [ ]:
I96 = resize(img_as_float(vie.msipimg(1, None, "gray")), (96, 96), order=1, anti_aliasing=True)
Xp = I96.reshape(-1, 2)                        # S×2：各行が水平隣接画素ペア (x_0, x_1)
exSx = np.cov(Xp.T, bias=True)
exSy = np.cov((Xp @ Aex.T).T, bias=True)
print("exSx =\n", exSx)
print("exSy =\n", exSy)
txtSx = np.array([[0.0320, 0.0277], [0.0277, 0.0322]])   # 教科書の値（図 6.1 の説明）
txtSy = np.array([[0.0597, -0.0001], [-0.0001, 0.0044]])
print("教科書の値：\ntxtSx =\n", txtSx, "\ntxtSy =\n", txtSy)
print(f"差の最大値：{np.max(np.abs(np.r_[np.round(exSx, 4) - txtSx, np.round(exSy, 4) - txtSy])):.4f}")

## 基底ベクトルと座標
$\mathbf{B}=\mathbf{A}^{-1}$ の列 $\mathbf{b}_0,\mathbf{b}_1$ が基底ベクトル。係数 $\mathbf{y}$ は新しい軸（基底ベクトルの方向）での座標。

In [ ]:
r2 = np.sqrt(2)
xs = np.array([[r2, r2], [r2, 2 * r2], [2 * r2, r2]]).T   # 列が x
ys = A @ xs                                    # 列が係数
recon = B @ ys                                 # 元に戻る
print("xs =\n", xs)
print("ys =\n", ys)
print("recon =\n", recon)
assert np.array_equal(np.round(ys), [[2, 3, 3], [0, 1, -1]])   # スライドの値

基底ベクトルの調べ方：係数を一つだけ 1，残りを 0 として逆変換する。

In [ ]:
b0 = B @ [1, 0]
b1 = B @ [0, 1]
print("b0 =", b0)
print("b1 =", b1)

## 基底画像（2×2 のハール変換）
分離処理では，順変換 $\mathbf{Y}=\mathbf{A}\mathbf{X}\mathbf{A}^\top$，逆変換 $\mathbf{X}=\mathbf{B}\mathbf{Y}\mathbf{B}^\top$。

In [ ]:
X2 = np.array([[0, 2], [4, 6]])
Y2 = A @ X2 @ A.T
X2r = B @ Y2 @ B.T
print("Y2 =\n", Y2)
print("X2r =\n", X2r)
assert np.array_equal(np.round(Y2), [[6, 2], [4, 0]])   # スライドの値

基底画像 $\mathsf{b}_{m_\mathrm{v},m_\mathrm{h}}=\mathbf{b}_{m_\mathrm{v}}\mathbf{b}_{m_\mathrm{h}}^\top$（係数を一つだけ 1 にして逆変換したもの）。画像は基底画像の重み付け和になる。

In [ ]:
Bimg = {}
for mv in range(2):
    for mh in range(2):
        E = np.zeros((2, 2))
        E[mv, mh] = 1
        Bimg[mv, mh] = B @ E @ B.T
        print(f"b_{{{mv},{mh}}} =\n", Bimg[mv, mh])
sumcheck = Y2[0, 0] * Bimg[0, 0] + Y2[0, 1] * Bimg[0, 1] + Y2[1, 0] * Bimg[1, 0] + Y2[1, 1] * Bimg[1, 1]
print("sumcheck =\n", sumcheck)

## 例題「二次元DCT」
DCT 行列 $\mathbf{C}_M$ を教科書の定義どおりに作る。

$$[\mathbf{C}_M]_{m,n}=\alpha_m\cos\frac{m(2n+1)\pi}{2M},\quad \alpha_0=1/\sqrt{M},\ \alpha_m=\sqrt{2/M}\ (m\neq0)$$

単位行列の各列を `scipy.fft.dct(..., norm="ortho")` で変換した行列（関数 `dctmtx`）と一致する。

In [ ]:
def dctmtx(M):
    """M 点 DCT 行列 C_M（単位行列の各列の DCT）"""
    return fft.dct(np.eye(M), norm="ortho", axis=0)


M = 4
mm, nn = np.meshgrid(np.arange(M), np.arange(M), indexing="ij")
am = np.r_[1 / np.sqrt(M), np.sqrt(2 / M) * np.ones(M - 1)][:, None]   # α_m
C4 = am * np.cos(mm * (2 * nn + 1) * np.pi / (2 * M))
err_dctmtx = np.linalg.norm(C4 - dctmtx(M), 2)
print("C4 =\n", C4)
print("err_dctmtx =", err_dctmtx)

教科書の例題：$4\times4$ 配列の二次元 DCT（分離処理 $\mathbf{Y}=\mathbf{C}_4\mathbf{X}\mathbf{C}_4^\top$）。縦方向に $\mathbf{U}=\mathbf{C}_4\mathbf{X}$，続いて横方向に $\mathbf{Y}=\mathbf{U}\mathbf{C}_4^\top$ と 2 段階で計算してもよい。

In [ ]:
X4 = np.array([[4, 4, 6, 4], [4, 6, 4, 2], [6, 4, 2, 4], [4, 2, 4, 4]])
U4 = C4 @ X4                                   # 縦方向の変換
Y4 = U4 @ C4.T
print("Y4 =\n", Y4)
txtY4 = np.array([[16.0000, 1.3066, 0, 0.5412], [1.3066, 0, -2.3890, 0],
                  [0, -2.3890, 0, 2.0719], [0.5412, 0, 2.0719, 0]])
match_dct = np.array_equal(np.round(Y4, 4), txtY4)  # 教科書の解答と一致するか
print("match_dct =", match_dct)
assert match_dct, "例題「二次元DCT」の値が教科書と一致しません"

エネルギー（二乗和）は保存され，直流成分 $y[0,0]$ に集中する。

In [ ]:
energy = np.array([np.sum(X4**2), np.sum(Y4**2), Y4[0, 0]**2 / np.sum(Y4**2)])
print("energy =", energy)
print(f"直流成分の割合 {100 * energy[2]:.0f} %")
assert f"{100 * energy[2]:.0f}" == "91"        # スライドの値

## 基底画像による展開の図
例題の配列 $\mathsf{x}$ を，DCT の基底画像 $\mathsf{b}_{m_\mathrm{v},m_\mathrm{h}}=\mathbf{b}_{m_\mathrm{v}}\mathbf{b}_{m_\mathrm{h}}^\top$（$\mathbf{b}_m=[\mathbf{C}_4^\top]_{:,m}$）の線形結合

$$\mathsf{x}=\sum_{m_\mathrm{v}=0}^{3}\sum_{m_\mathrm{h}=0}^{3}y[m_\mathrm{v},m_\mathrm{h}]\,\mathsf{b}_{m_\mathrm{v},m_\mathrm{h}}$$

で表す。スライドの「画像変換の基礎」で使う横一列の図を作る：左から $\mathsf{x}$，$\mathsf{b}_{0,0}$，$\mathsf{b}_{0,1}$，$\mathsf{b}_{0,2}$，$\mathsf{b}_{3,3}$，係数配列 $\mathsf{y}$（絶対値を対数で強調）。基底画像は 0 を灰色，正を白，負を黒で表す。記号はスライド側（TikZ）で付けるので，各タイルの左端の位置 `left` はスライドと合わせてある。

In [ ]:
Bc = C4.T                                      # 列が基底ベクトル b_m


def bimg(mv, mh):
    """基底画像 b_{mv,mh}"""
    return np.outer(Bc[:, mv], Bc[:, mh])


def togray(b):
    """0 を灰色，正を白，負を黒に"""
    return 0.5 + b / (2 * np.max(np.abs(b)))


recon4 = np.zeros((M, M))
for mv in range(M):
    for mh in range(M):
        recon4 = recon4 + Y4[mv, mh] * bimg(mv, mh)   # 基底画像の重み付け和
err_expand = np.max(np.abs(recon4 - X4))       # 元の配列に戻る
print("err_expand =", err_expand)
tiles = [X4 / X4.max(), togray(bimg(0, 0)), togray(bimg(0, 1)), togray(bimg(0, 2)),
         togray(bimg(3, 3)), rescale_intensity(np.log(1 + np.abs(Y4)), out_range=(0.0, 1.0))]
left = [0, 108, 216, 324, 472, 620]            # 各タイルの左端（画素）
tw = 68                                        # タイルの幅（64 画素＋枠 2 画素×2）
canvas = np.ones((tw, left[-1] + tw))
for k, tile in enumerate(tiles):
    t = np.pad(np.kron(tile, np.ones((16, 16))), 2, constant_values=0.3)   # 4×4 → 64×64，濃い灰色の枠
    canvas[:, left[k]:left[k] + tw] = t
fig = plt.figure(figsize=(canvas.shape[1] / 100, canvas.shape[0] / 100))   # 1 画素を画面の 1 画素に
show(fig.add_axes((0, 0, 1, 1)), canvas)
plt.show()

## 8 点 DCT の基底ベクトルと基底画像
基底ベクトル $\mathbf{b}_m=[\mathbf{C}_8^\top]_{:,m}$（$\mathbf{C}_8$ の $m$ 行目）を教科書の図 6.2 (a)（$M=4$）と同じ体裁で描く。

In [ ]:
C8 = dctmtx(8)
fig, axs = plt.subplots(4, 2, figsize=(5.6, 5.2), layout="constrained")
for m, ax in enumerate(axs.flat):
    ml, sl, bl = ax.stem(np.arange(8), C8[m], basefmt="-")
    plt.setp(ml, color=cmain, ms=4)
    plt.setp(sl, color=cmain)
    plt.setp(bl, color=cgray)
    ax.set(ylim=(-0.6, 0.6), xlim=(-0.5, 7.5), xticks=range(8), yticks=[-0.5, 0, 0.5])
    ax.set_title(rf"$\mathbf{{b}}_{{{m}}}$")
fig.supxlabel("$n$", fontsize=13)
plt.show()

基底画像（$8\times8$ 通り）を並べる（灰色の枠で区切る）。

In [ ]:
Mos = 0.5 * np.ones((8 * 9 + 1, 8 * 9 + 1))
for mv in range(8):
    for mh in range(8):
        Bm = np.outer(C8[mv], C8[mh])          # 基底画像 = 基底ベクトルの外積
        r, c = mv * 9 + 1, mh * 9 + 1
        Mos[r:r + 8, c:c + 8] = 0.5 + Bm / (2 * np.max(np.abs(Bm)))
fig = plt.figure(figsize=(6 * Mos.shape[1] / 100, 6 * Mos.shape[0] / 100))   # 1 画素を画面の 6×6 画素に
show(fig.add_axes((0, 0, 1, 1)), Mos)
plt.show()

## 例題「二次元DHT」
DHT 行列を教科書の定義（クロネッカー積による再帰）どおりに作る（関数 `dhtmtx`）。

$$\mathbf{H}_{2^J}=\frac{1}{\sqrt{2}}\begin{pmatrix}\mathbf{H}_{2^{J-1}}\otimes(1\ \ 1)\\\mathbf{I}_{2^{J-1}}\otimes(1\ -1)\end{pmatrix},\quad \mathbf{H}_1=1$$

In [ ]:
def dhtmtx(J):
    """教科書の定義「離散ハール変換（DHT）」の行列 H_{2^J} を再帰で作る"""
    if J == 0:
        return np.ones((1, 1))                 # H_1 = 1
    Hp = dhtmtx(J - 1)
    return np.vstack([np.kron(Hp, [1, 1]), np.kron(np.eye(2 ** (J - 1)), [1, -1])]) / np.sqrt(2)


H4 = dhtmtx(2)
H4txt = np.array([[1, 1, 1, 1], [1, 1, -1, -1], [r2, -r2, 0, 0], [0, 0, r2, -r2]]) / 2   # 教科書の解答の H_4
err_H4 = np.linalg.norm(H4 - H4txt, 2)
orth_H4 = np.linalg.norm(H4 @ H4.T - np.eye(4), 2)   # 正規直交
print("H4 =\n", H4)
print("err_H4 =", err_H4, " orth_H4 =", orth_H4)
assert err_H4 < 1e-12 and orth_H4 < 1e-12

例題「二次元DCT」と同じ配列に，分離処理 $\mathbf{Y}=\mathbf{H}_4\mathbf{X}\mathbf{H}_4^\top$ を適用する。

In [ ]:
Y4dht = H4 @ X4 @ H4.T
print("Y4dht =\n", Y4dht)
txtY4dht = np.array([[16.0000, 1.0000, 0.7071, 0.7071], [1.0000, 0, -2.1213, 2.1213],
                     [0.7071, -2.1213, 1.0000, 0], [0.7071, 2.1213, 0, -1.0000]])
match_dht = np.array_equal(np.round(Y4dht, 4), txtY4dht)
print("match_dht =", match_dht)
assert match_dht, "例題「二次元DHT」の値が教科書と一致しません"

## 例題「ブロック処理」
教科書の例題「標準化と最小最大正規化」の $4\times6$ 配列 $\mathsf{x}$ を $2\times2$ のブロックに分け，各ブロック $\mathbf{X}_b$ に二次元 DCT $\mathbf{C}_2\mathbf{X}_b\mathbf{C}_2^\top$ を施す。逆変換 $\mathbf{C}_2^\top\mathbf{Y}_b\mathbf{C}_2$ で元に戻る。ブロックへの分割は scikit-image の `view_as_blocks` で行い，各ブロックの結果を並べ直す関数 `unblock` をここで定義して使う。

In [ ]:
def unblock(Yb):
    """view_as_blocks の形（ブロック行, ブロック列, 行, 列）の配列を一枚の配列に並べ直す"""
    nbv, nbh, bv, bh = Yb.shape
    return Yb.transpose(0, 2, 1, 3).reshape(nbv * bv, nbh * bh)


xblk = np.array([[5, 5, 7, 6, 7, 2], [6, 2, 3, 2, 7, 2], [2, 2, 4, 4, 4, 6], [4, 4, 4, 4, 1, 2]])
C2 = dctmtx(2)                                 # = [1 1; 1 -1]/sqrt(2) = H_2
print("C2 =\n", C2)
yblk = unblock(C2 @ view_as_blocks(xblk.astype(float), (2, 2)) @ C2.T)   # 各ブロックに C_2 X_b C_2^T
print("yblk =\n", yblk)
txtyblk = np.array([[9, 2, 9, 1, 9, 5], [1, -2, 4, 0, 0, 0], [6, 0, 8, 0, 6.5, -1.5], [-2, 0, 0, 0, 3.5, -0.5]])
match_blk = np.array_equal(np.round(yblk, 4), txtyblk)
print("match_blk =", match_blk)
assert match_blk, "例題「ブロック処理」の値が教科書と一致しません"
xblk_r = unblock(C2.T @ view_as_blocks(yblk, (2, 2)) @ C2)
err_blk = np.max(np.abs(xblk_r - xblk))        # 元に戻る
print("err_blk =", err_blk)

## 画像の 8×8 ブロック DCT
各ブロックの係数（絶対値の対数）を表示すると，各ブロックの左上（低周波）に大きな値が集まる。

In [ ]:
Yb = unblock(fft.dctn(view_as_blocks(X, (8, 8)), axes=(2, 3), norm="ortho"))
fig = plt.figure(figsize=(5.12, 5.42))         # 1 画素を画面の 2×2 画素に（上は題名の余白）
show(fig.add_axes((0, 0, 1, 512 / 542)), rescale_intensity(np.log(1 + 50 * np.abs(Yb)), out_range=(0.0, 1.0)),
           "8×8 ブロック DCT の係数")         # 小さな係数も見えるよう強調
plt.show()

係数を大きい順に並べると，エネルギーの 99% は何 % の係数で表せるか。

In [ ]:
e = np.sort(Yb.ravel()**2)[::-1]
k99 = (np.argmax(np.cumsum(e) >= 0.99 * np.sum(e)) + 1) / e.size
print(f"k99 = {k99:.4f}（{100 * k99:.1f} %）")

各ブロックで大きい係数だけ（全体の 10%）残して逆変換してみる。

In [ ]:
thr = np.percentile(np.abs(Yb), 90)
Yk = Yb * (np.abs(Yb) >= thr)
Xk = unblock(fft.idctn(view_as_blocks(Yk, (8, 8)), axes=(2, 3), norm="ortho"))
Xc = np.clip(Xk, 0, 1)
psnr10 = peak_signal_noise_ratio(X, Xc, data_range=1)   # ピーク値 1 の PSNR [dB]
print(f"psnr10 = {psnr10:.4f} dB")
fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
show(axs[0], X, "原画像")
show(axs[1], Xc, f"係数 10% で再構成（{psnr10:.1f} dB）")
plt.show()

## ハール変換による信号の解析と合成
$x[n]=(3,1,3,1,5,3)$ に対し，隣同士を足して 1/2 倍（縮小近似成分），左隣を引いて 1/2 倍（縮小詳細成分），2 点に 1 点を残す。

In [ ]:
x = np.array([3, 1, 3, 1, 5, 3])
s = x[1:] + x[:-1]                             # 隣同士を足す
d = x[1:] - x[:-1]                             # 左隣を引く
a = s[::2] / 2                                 # 縮小近似成分
dd = d[::2] / 2                                # 縮小詳細成分
print("s =", s)
print("d =", d)
print("a =", a)
print("dd =", dd)
assert (np.array_equal(s, [4, 4, 4, 6, 8]) and np.array_equal(d, [-2, 2, -2, 4, -2])
        and np.array_equal(a, [2, 2, 4]) and np.array_equal(dd, [-1, -1, -1]))   # スライドの値

合成：近似成分は右隣にコピー（最近傍補間），詳細成分は符号反転して左隣にコピーして足す。

In [ ]:
up_a = np.kron(a, [1, 1])
up_d = np.kron(dd, [-1, 1])
xr = up_a + up_d
print("up_a =", up_a)
print("up_d =", up_d)
print("xr =", xr)
print(np.array_equal(xr, x))
assert np.array_equal(up_a, [2, 2, 2, 2, 4, 4]) and np.array_equal(up_d, [1, -1, 1, -1, 1, -1])   # スライドの値

## フィルタバンク表現
分析器 $H_0(z)=\frac12(1+z^{-1})$，$H_1(z)=\frac12(1-z^{-1})$ とダウンサンプラ，アップサンプラと合成器 $F_0(z)=1+z^{-1}$，$F_1(z)=-1+z^{-1}$ で構成する。出力は入力を 1 サンプル遅らせたものに一致する（完全再構成）。

In [ ]:
xf = np.r_[x, 0, 0].astype(float)              # 最後の出力を見るため零を足す
v0 = signal.lfilter([1 / 2, 1 / 2], 1, xf)
v1 = signal.lfilter([1 / 2, -1 / 2], 1, xf)
y0 = v0[1::2]                                  # ↓2（奇数番目を残す）：縮小近似成分
y1 = v1[1::2]                                  # 縮小詳細成分
u0 = np.zeros_like(xf)
u0[1::2] = y0                                  # ↑2
u1 = np.zeros_like(xf)
u1[1::2] = y1
w = signal.lfilter([1.0, 1.0], 1, u0) + signal.lfilter([-1.0, 1.0], 1, u1)   # = [0 x]（1 サンプル遅延）
print("y0 =", y0)
print("y1 =", y1)
print("w =", w)
assert np.array_equal(w[:7], [0, 3, 1, 3, 1, 5, 3])   # スライドの値

## 多相行列表現（等価変換）
偶数番目 $x_0=x[2m]$ と奇数番目 $x_1=x[2m+1]$ の組に対する分析器 $\mathbf{E}$ と合成器 $\mathbf{R}$。$\mathbf{R}\mathbf{E}=\mathbf{I}$ で完全に元に戻る。

In [ ]:
E = np.array([[1, 1], [-1, 1]]) / 2
R = np.array([[1, -1], [1, 1]])
RE = R @ E
coef = E @ x.reshape(-1, 2).T                  # 列が (y0, y1)
print("E =\n", E)
print("R =\n", R)
print("RE =\n", RE)
print("coef =\n", coef)

## まとめ
- 自然画像は隣接画素の相関が強い。信号変換で相関を減らし，係数を疎（スパース）にできる
- 信号は基底ベクトル（$\mathbf{A}^{-1}$ の列）の線形結合，画像は基底画像の線形結合で表せる
- DCT は低周波にエネルギーを集中させる。DHT は加減算で計算でき，ブロック処理で大きな画像にも適用できる
- ハール変換はフィルタバンク（分析器と合成器）で実現でき，ブロック変換と等価

© Copyright, Shogo MURAMATSU, All rights reserved.